# AniRoute — train the road-surface model (Model A) on Google Colab

This notebook **fine-tunes a COCO-pretrained YOLO11 model** (transfer learning) to detect
`pothole` and `severe_road_damage`, runs the optimisation experiments **E0–E4 (+ E7)**, picks
the best run on the *validation* split and evaluates it once on the *test* split.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.
Each training run takes roughly 15–40 minutes on a free T4 depending on dataset size.

You do **not** need to label anything: the dataset comes from Roboflow Universe (already labelled).

## 1 · Check the GPU

In [ ]:
!nvidia-smi

## 2 · Get the AniRoute code
Clone your GitHub repository (or upload a zip of the project and unzip it to `/content/AniRoute`).

In [ ]:
REPO_URL = "https://github.com/Nezraaaa/AniRoute.git"   # change if your repository is elsewhere
BRANCH = "feat/cv-final-project"
!git clone -b {BRANCH} {REPO_URL} /content/AniRoute || echo "Clone failed: upload the project zip instead"
%cd /content/AniRoute

## 3 · (Optional) keep results in Google Drive
Colab deletes files when the session ends. Mounting Drive lets you resume if the runtime disconnects.

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/aniroute_runs
    !ln -sfn /content/drive/MyDrive/aniroute_runs /content/AniRoute/ml/runs

## 4 · Install the training dependencies

In [ ]:
!pip install -q -r ml/requirements.txt roboflow

## 5 · Download the datasets (already labelled, YOLOv8 format)

**Option A — Roboflow API (recommended).** In Roboflow, fork the two datasets into your workspace
(see `docs/DATASET.md`), open each one → *Download dataset* → *YOLOv8* → *Show download code*, and copy
the workspace, project and version values below. Your API key is under *Settings → API keys*.

**Option B — upload zips.** Download each dataset as a YOLOv8 zip and upload it to `ml/data/raw/`,
then unzip with `!unzip -q ml/data/raw/<file>.zip -d ml/data/raw/<name>`.

In [ ]:
ROBOFLOW_API_KEY = ""   # paste your key, or leave empty and use Option B
DATASETS = [
    # (workspace, project, version, folder name)
    ("YOUR-WORKSPACE", "pothole-dataset", 1, "potholes"),
    ("YOUR-WORKSPACE", "rdd2022", 1, "rdd2022"),
]
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for workspace, project, version, name in DATASETS:
        rf.workspace(workspace).project(project).version(version).download("yolov8", location=f"ml/data/raw/{name}")
!ls ml/data/raw

## 6 · Merge, remap classes and check for leakage
`prepare_dataset.py` maps every source class to ours, drops unreadable images, removes near-duplicate
images from val/test that look like training images, writes dataset statistics and figures to
`ml/reports/`, and copies 6 held-out test images to `backend/samples/` for the demo.

Adjust `--map` to the class names printed for your datasets. RDD2022: `D40` = pothole,
`D00/D10/D20` = cracks → `severe_road_damage`. `--max-images` keeps training time reasonable.

In [ ]:
!python ml/scripts/prepare_dataset.py \
    --source ml/data/raw/potholes --source ml/data/raw/rdd2022 \
    --map pothole=pothole potholes=pothole D40=pothole \
          D00=severe_road_damage D10=severe_road_damage D20=severe_road_damage \
    --max-images 2500

## 7 · Optimisation experiments
Run them one at a time. **E0 is the baseline**; each later run changes one thing.

| ID | Change |
| --- | --- |
| E0 | Baseline: COCO-pretrained YOLO11n, defaults, 50 epochs, 640 px |
| E1 | 100 epochs with early stopping |
| E2 | YOLO11s (bigger model) |
| E3 | 800 px input (small potholes) |
| E4 | Lighting/rain augmentation (hsv_v 0.5, ±5° rotation, mixup 0.1) |
| E7 | *(optional)* same network trained **from scratch** — shows the value of pretraining |

E5 (threshold tuning) and E6 (CLAHE on/off) need no training; `evaluate.py compare` computes them.
If time is short, run E0, E1 and E2 at least.

In [ ]:
!python ml/train.py --experiment E0 --device 0

In [ ]:
!python ml/train.py --experiment E1 --device 0

In [ ]:
!python ml/train.py --experiment E2 --device 0

In [ ]:
!python ml/train.py --experiment E3 --device 0

In [ ]:
!python ml/train.py --experiment E4 --device 0

In [ ]:
# Optional, slow: shows what transfer learning contributes
# !python ml/train.py --experiment E7 --device 0

## 8 · Compare runs on the validation split and pick the best
Speeds measured here are Colab CPU speeds; re-run `python ml/evaluate.py latency` on the laptop
you will present with for the final speed table.

In [ ]:
import glob
runs = " ".join(sorted(d for d in glob.glob("ml/runs/E*") if not d.endswith("E7"))) + (" ml/runs/E7" if glob.glob("ml/runs/E7/weights/best.pt") else "")
!python ml/evaluate.py compare --runs {runs}
!cat ml/reports/optimization_table.md

## 9 · Deploy the selected run and evaluate it once on the test split

In [ ]:
import json, shutil
selected = json.load(open("ml/reports/optimization.json"))["selected"]
shutil.copy(f"ml/runs/{selected}/weights/best.pt", "backend/models/aniroute-hazards-v1.pt")
print("deployed", selected)
!python ml/evaluate.py test --run-dir ml/runs/{selected}
!python ml/scripts/preprocess_demo.py

## 10 · Download everything you need on the laptop
The zip contains the trained weights, every report and figure, the demo samples and the training runs'
curves. Unzip it into the project folder on your laptop (it keeps the same paths).

In [ ]:
!zip -qr aniroute_results.zip backend/models/aniroute-hazards-v1.pt ml/reports backend/samples \
    ml/runs/*/results.png ml/runs/*/results.csv ml/runs/*/args.yaml ml/runs/*/aniroute_experiment.json \
    ml/runs/*/weights/best.pt ml/data/aniroute/data.yaml
from google.colab import files
files.download("aniroute_results.zip")

## Back on the laptop
1. Unzip `aniroute_results.zip` into the project folder.
2. `python ml/export.py` (ONNX/OpenVINO for faster CPU inference) and `python ml/evaluate.py latency`.
3. Optional: `python ml/scripts/make_demo_video.py --video <your phone recording>.mp4` for the demo drive.
4. Start the app (`scripts/start.ps1`) — the Evaluation page now shows the real numbers.
5. `python presentation/build_deck.py` rebuilds the slides with the final figures.